# Evaluation : 

RAGAS metrics + draft quality eval

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import pandas as pd
from triage.agents.graph import build_graph

app = build_graph(model_dir="../models")

# Load the processed data directly for sampling
df = pd.read_csv('../data/processed/tickets_clean.csv')

eval_df = df.sample(20, random_state=42).reset_index(drop=True)

questions, answers, contexts, ground_truths = [], [], [], []

for _, row in eval_df.iterrows():
    result = app.invoke({"ticket_text": row['text_input']})
    questions.append(row['text_input'])
    answers.append(result['draft_response'])
    contexts.append([c['answer'] for c in result['retrieved_context']])
    ground_truths.append(row['answer'])

print(len(questions))

20


In [3]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

# 1. Import the LangChain OpenAI models and Ragas wrappers
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# 2. Initialize and wrap the models
llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

eval_dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths,
})

# 3. Explicitly pass llm and embeddings into evaluate
results = evaluate(
    eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=llm,
    embeddings=embeddings
)

print(results)


C:\Users\jenil\AppData\Local\Temp\ipykernel_3176\3284309038.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\jenil\AppData\Local\Temp\ipykernel_3176\3284309038.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\jenil\AppData\Local\Temp\ipykernel_3176\3284309038.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import c

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

{'faithfulness': 0.9400, 'answer_relevancy': 0.4329, 'context_precision': 1.0000}


In [4]:
print("Classifier accuracy: 50% (XGBoost, TF-IDF features, 10-class queue prediction)")
print("Macro avg F1: 0.44 — weaker on minority classes (General Inquiry, Sales and Pre-Sales)")

Classifier accuracy: 50% (XGBoost, TF-IDF features, 10-class queue prediction)
Macro avg F1: 0.44 — weaker on minority classes (General Inquiry, Sales and Pre-Sales)


In [1]:
from triage.db.database import init_db, get_session
from triage.db import crud

init_db()  # creates data/triage.db and the tickets table
session = get_session()

t = crud.create_ticket(session, "test@example.com", "Test issue", "My internet keeps dropping")
print(t.id, t.status)

fetched = crud.get_ticket(session, t.id)
print(fetched.subject)

c170e0ab processing
Test issue
